# 01 — Analytical Solutions for Shell and Cavity Modes

This notebook explores the closed-form solutions implemented in
`browntone.analytical` to build intuition about the frequency ranges
we expect for the abdominal cavity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from browntone.analytical.shell_vibration import cylindrical_shell_frequencies
from browntone.analytical.acoustic_modes import cylindrical_cavity_modes
from browntone.utils.materials import get_material, get_fluid
from browntone.utils.constants import INFRASOUND_UPPER_HZ
from browntone.postprocess.visualization import set_publication_style, COLOURS

set_publication_style()
print("Setup complete.")

## Shell Modes

Compute the natural frequencies of a cylindrical shell with soft-tissue properties.

In [ ]:
tissue = get_material("soft-tissue")

modes = cylindrical_shell_frequencies(
    radius_m=0.15,
    length_m=0.30,
    thickness_m=0.015,
    material=tissue,
    m_max=5,
    n_max=10,
)

print(f"Number of modes: {len(modes)}")
print(f"Fundamental frequency: {modes[0].frequency_hz:.2f} Hz")
print(f"First 10 modes:")
for i, m in enumerate(modes[:10]):
    print(f"  Mode {i+1}: f = {m.frequency_hz:.2f} Hz  (m={m.m}, n={m.n})")

## Acoustic Cavity Modes

Compute the rigid-wall acoustic modes of a cylindrical cavity.

In [ ]:
fluid = get_fluid("abdominal-fluid")

acoustic_modes = cylindrical_cavity_modes(
    radius_m=0.15,
    length_m=0.30,
    sound_speed_m_s=fluid.sound_speed_m_s,
)

print(f"Number of acoustic modes: {len(acoustic_modes)}")
print(f"First acoustic mode: {acoustic_modes[0].frequency_hz:.2f} Hz")
print(f"\nFirst 10 acoustic modes:")
for i, m in enumerate(acoustic_modes[:10]):
    print(f"  Mode {i+1}: f = {m.frequency_hz:.2f} Hz  {m.description}")

## Comparison: Are any modes in the infrasound range?

In [ ]:
shell_freqs = np.array([m.frequency_hz for m in modes[:20]])
acoustic_freqs = np.array([m.frequency_hz for m in acoustic_modes[:20]])

fig, ax = plt.subplots(figsize=(6, 3))

ax.scatter(shell_freqs, np.ones_like(shell_freqs), color=COLOURS["primary"],
           s=50, label="Shell modes", zorder=5)
ax.scatter(acoustic_freqs, 2 * np.ones_like(acoustic_freqs), color=COLOURS["secondary"],
           s=50, label="Acoustic modes", zorder=5)

ax.axvspan(0, INFRASOUND_UPPER_HZ, alpha=0.15, color="gray", label="Infrasound range")

ax.set_xlabel("Frequency (Hz)")
ax.set_yticks([1, 2])
ax.set_yticklabels(["Shell", "Acoustic"])
ax.legend(loc="upper right")
ax.set_title("Natural frequencies vs infrasound range")

plt.tight_layout()
plt.show()